#### Architectural Rationale
While our One Big Tables (OBTs) provide granular, denormalized data foundations, querying them directly for high-level business intelligence is computationally inefficient and cognitively overwhelming. 

In this phase, we extract highly aggregated, domain-specific **Data Marts** from the OBTs. This decouples the heavy data engineering logic from the frontend analytical layer, ensuring our BI dashboards and causal inference models run on lightweight, tightly scoped, and heavily optimized datasets.

### 0. Read data & Import lib 

In [ ]:
import duckdb

Frontend_Traffic_OBT = duckdb.sql('SELECT * FROM "../dataset/processed/Frontend_Traffic_OBT.parquet";')
Backend_Sales_OBT = duckdb.sql('SELECT * FROM "../dataset/processed/Backend_Sales_OBT.parquet";')

### Create Data Marts

#### 3.1 Financial Summary Mart (`mart_financial_summary`)

***Business Logic & Metric Definitions***
To support executive-level BI dashboards, this Data Mart aggregates item-level transactional data from the `Backend_Sales_OBT` into a quarterly financial ledger. We explicitly decouple "vanity metrics" from true profitability indicators to prevent skewed revenue reporting:

1. **Top-Line vs. Bottom-Line:** We isolated `GMV` (gross merchandise value) from `Net Revenue`. The true cash-in-bank (`Net Revenue`) and `Gross Profit` are strictly calculated by omitting 'Returned' and 'Cancelled' item statuses.
2. **Unit Economics:** Computed `AOV` (Average Order Value) and `ARPU` (Average Revenue Per User) to track customer monetization efficiency over time.
3. **Operational Health:** Integrated `Return Rate (%)` alongside `Gross Margin (%)` to monitor the direct impact of reverse logistics on overall profitability.

**Data Engineering Action:**
Implemented a CTE (Common Table Expression) architecture to pre-aggregate foundational base metrics. This ensures complex downstream health ratios (e.g., margins, ARPU) are executed cleanly without repetitive `CASE WHEN` evaluations, significantly optimizing query execution plans.

In [3]:
mart_financial_summary= duckdb.sql("""
WITH Base_Metrics AS (
    SELECT
        CONCAT(YEAR(order_created_at), '-Q', QUARTER(order_created_at)) AS year_quarter,
        CONCAT('Q', QUARTER(order_created_at)) AS quarter,
        YEAR(order_created_at) AS year,

        COUNT(DISTINCT user_id) AS active_buyers,
        COUNT(DISTINCT order_id) AS total_orders,
        COUNT(order_items_id) AS total_items_sold,

        SUM(CASE WHEN items_status = 'Returned' THEN 1 ELSE 0 END) AS returned_items_count,

        SUM(sale_price) AS GMV,
        SUM(CASE WHEN items_status = 'Returned' THEN sale_price ELSE 0 END) AS return_amount,
        
        SUM(CASE WHEN items_status NOT IN ('Returned', 'Cancelled') THEN sale_price ELSE 0 END) AS net_revenue,
        SUM(CASE WHEN items_status NOT IN ('Returned', 'Cancelled') THEN cost ELSE 0 END) AS COGS

    FROM Backend_Sales_OBT
    WHERE user_id IS NOT NULL
    GROUP BY YEAR(order_created_at), QUARTER(order_created_at)
)

SELECT
    year_quarter,
    quarter,
    year,
    active_buyers,
    total_orders,
    total_items_sold,

    ROUND(return_amount, 2) AS return_amount,
    ROUND(GMV, 2) AS GMV,
    ROUND(net_revenue, 2) AS net_revenue,
    ROUND(COGS, 2) AS COGS,
    ROUND(net_revenue - COGS, 2) AS gross_profit,

    ROUND(100.0 * (net_revenue - COGS) / NULLIF(net_revenue, 0), 2) AS gross_margin_pct,
    ROUND(GMV / NULLIF(total_orders, 0), 2) AS AOV,
    ROUND(net_revenue / NULLIF(active_buyers, 0), 2) AS ARPU,
    ROUND(100.0 * returned_items_count / NULLIF(total_items_sold, 0), 2) AS return_rate_pct,

FROM Base_Metrics
ORDER BY Year, Quarter
""")
mart_financial_summary.to_parquet('../dataset/processed/mart_financial_summary.parquet')


#### 3.2 Mart RFM Segmentation(`mart_rfm_segmentation`)

***Business Logic & Feature Engineering***
To move beyond aggregate financial metrics, we engineered an RFM (Recency, Frequency, Monetary) segmentation mart to map customer lifecycle states. 

**Engineering Highlights:**
1. **Dynamic Cohort Scoring:** Instead of computing a single static lifetime RFM score, we aggregated data at the `Year_Quarter` level. By applying `NTILE(5)` window functions partitioned by time (`PARTITION BY Year_Quarter`), we evaluate customers relative to their concurrent cohort. This produces highly valuable Panel Data, tracking how user segments migrate over time.
2. **Defensive Filtering:** Strictly excluded 'Cancelled' and 'Returned' statuses from the RFM evaluation to prevent artificial inflation of customer lifetime value (LTV).
3. **Pipeline Optimization:** Consolidated the absolute value calculation, percentile ranking, and heuristic segmentation (the `CASE WHEN` rules) into a single optimized CTE pipeline, avoiding the creation of multiple volatile intermediate tables.

In [4]:
mart_rfm_segmentation= duckdb.sql("""
WITH Max_Date AS (
    SELECT MAX(order_created_at) AS global_max_date
    FROM Backend_Sales_OBT
),
RFM_Base AS (
    SELECT 
        CONCAT(YEAR(order_created_at), '-Q', QUARTER(order_created_at)) AS Year_Quarter,
        user_id,
        DATEDIFF('day', MAX(order_created_at), (SELECT global_max_date FROM Max_Date)) AS recency,
        COUNT(DISTINCT order_id) AS frequency,
        ROUND(SUM(sale_price), 2) AS monetary
    FROM Backend_Sales_OBT
    WHERE items_status NOT IN ('Cancelled','Returned') 
      AND user_id IS NOT NULL 
      AND order_created_at >= (SELECT global_max_date - INTERVAL 1 YEAR FROM Max_Date)
    GROUP BY user_id, YEAR(order_created_at), QUARTER(order_created_at)
),
RFM_Scores AS (
    SELECT 
        *,
        NTILE(5) OVER (PARTITION BY Year_Quarter ORDER BY recency DESC) AS r_score,
        NTILE(5) OVER (PARTITION BY Year_Quarter ORDER BY frequency ASC) AS f_score,
        NTILE(5) OVER (PARTITION BY Year_Quarter ORDER BY monetary ASC) AS m_score
    FROM RFM_Base
)
SELECT 
    *,
    CASE
        WHEN r_score >= 4 AND f_score >= 4 AND m_score >= 4 THEN 'Champions'
        WHEN r_score = 1 AND f_score >= 4 AND m_score >= 4 THEN 'Cant Lose Them' 
        WHEN r_score >= 3 AND f_score >= 3 AND m_score >= 3 THEN 'Loyalists'
        WHEN r_score >= 4 AND f_score = 1 THEN 'New Customers'
        WHEN r_score >= 3 AND f_score <= 3 THEN 'Potential Loyalists' 
        WHEN r_score IN (2, 3) AND f_score IN (2, 3) THEN 'Need Attention'
        WHEN r_score <= 2 AND f_score >= 2 AND m_score >= 2 THEN 'At Risk'
        WHEN r_score <= 2 AND f_score >= 4 AND m_score <= 2 THEN 'Lost Bargain Hunters'
        WHEN r_score <= 2 AND f_score <= 2 AND m_score <= 2 THEN 'Hibernating / Lost'
        ELSE 'Others' 
    END AS rfm_segment
FROM RFM_Scores
ORDER BY Year_Quarter, r_score DESC, f_score DESC, m_score DESC
""")

mart_rfm_segmentation.to_parquet('../dataset/processed/mart_rfm_segmentation.parquet')

#### 3.3 Mart Logistics Experience(`mart_logistics_experience`)

***Business Logic & Feature Engineering***
This Data Mart serves as the analytical foundation for our causal inference analysis regarding supply chain latency and customer return behaviors. We aggregated item-level fulfillment data up to the `order_id` grain to accurately reflect the end-user's actual delivery experience.

**Engineering Highlights:**
1. **True Fulfillment Latency:** Delivery duration is explicitly engineered as the time delta between the initial order creation and the `MAX(items_delivered_at)`. This accurately models the consumer perspective, where an order is only considered fulfilled when the *final* package arrives.
2. **Split Shipment Detection:** By extracting `COUNT(DISTINCT product_distribution_center_id)`, we inherently created a feature to track multi-warehouse split fulfillments. This allows downstream models to test if fragmented deliveries (receiving multiple packages on different days) degrade user experience and drive higher return rates.
3. **Causal Anchor:** Engineered a binary `return_flag` to serve as the target variable for our logistics latency distribution checks.

In [5]:
mart_logistics_experience= duckdb.sql("""
SELECT
    user_id,
    order_id,
    COUNT(DISTINCT order_items_id) AS total_items,
    COUNT(DISTINCT product_distribution_center_id) AS total_dc,
    DATEDIFF('hour',MIN(order_created_at),MAX(items_delivered_at)) AS total_fulfillment_time_hours,
    MAX(CASE WHEN orders_status = 'Returned' THEN 1 ELSE 0 END) AS return_flag
FROM
    Backend_Sales_OBT
WHERE
    user_id IS NOT NULL AND items_delivered_at IS NOT NULL 
GROUP BY
    order_id,user_id
ORDER BY user_id 
""")
mart_logistics_experience.to_parquet('../dataset/processed/mart_logistics_experience.parquet')

#### 3.4 Mart Experimentation Base(`mart_experimentation_base`)

***Business Logic & A/B Testing Readiness***
To bridge the gap between Data Engineering and Data Science, this Mart is structured explicitly for causal inference and A/B testing (e.g., evaluating conversion rates across different traffic acquisition channels). 

**Engineering Highlights:**
1. **Exposure vs. Outcome Separation:** We explicitly decoupled the "Exposure" layer (Frontend traffic source and page views) from the "Outcome" layer (Backend conversion and revenue). 
2. **Funnel Conversion Architecture:** Utilizing a `LEFT JOIN` from frontend traffic to backend sales allows us to engineer the `is_converted` binary flag and a continuous `total_spend` metric. This maps perfectly to standard A/B testing dual-metric frameworks (Conversion Rate + ARPU).
3. **Ghost-Traffic Filtering:** Filtered out `user_id = 0` (anonymous browsing traffic) during the aggregation phase to prevent the creation of a massive, skewed proxy-user, ensuring statistical denominator integrity for user-level experiments.

In [7]:
mart_experimentation_base= duckdb.sql("""
WITH frontend_agg AS (
    SELECT 
        user_id,
        ANY_VALUE(user_traffic_source) AS experiment_group,
        COUNT(event_id) AS total_page_view
    FROM Frontend_Traffic_OBT
    WHERE user_id != 0
    GROUP BY user_id
),
backend_agg AS (
    SELECT
        user_id,
        SUM(sale_price) AS total_spend
    FROM Backend_Sales_OBT
    GROUP BY user_id
)
SELECT
    f.user_id,
    f.experiment_group,
    f.total_page_view,
    CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS is_converted,
    COALESCE(b.total_spend, 0) AS total_spend

FROM frontend_agg AS f
LEFT JOIN backend_agg AS b 
    ON f.user_id = b.user_id
ORDER BY user_id
""")

mart_experimentation_base.to_parquet('../dataset/processed/mart_experimentation_base.parquet')

#### 3.5 Mart Impulse Penalty(`mart_impulse_penalty`)

***Business Logic & Causal EDA Foundation***
This Data Mart bridges the gap between top-of-funnel browsing behavior and bottom-of-funnel fulfillment outcomes to test the "Impulse Penalty" hypothesis. We engineered behavioral segments based on session duration and engagement depth to observe if quick, low-friction checkouts inherently drive higher return rates.

**Engineering Highlights:**
1. **Cross-Domain Join:** Seamlessly integrated web clickstream data (`clean_events`) with transactional outcomes (`Backend_Sales_OBT`) using a unified `user_id` key.
2. **Heuristic Segmentation:** Engineered intent cohorts (`Impulse Buyers`, `Heavy Browsers`, `Targeted Searchers`) dynamically via conditional bounds on `session_duration_minutes` and `total_page_views`.
3. **Pipeline Consolidation:** Compressed session aggregation, purchase filtering, intent tagging, and return rate calculation into a single execution plan using chained CTEs. This outputs a highly lightweight, chart-ready aggregate table.

In [1]:
mart_impulse_penalty = duckdb.sql("""
WITH session_aggregated AS (
    SELECT 
        session_id,
        MAX(user_id) AS user_id, 
        DATE_DIFF('minute', MIN(event_created_at), MAX(event_created_at)) AS session_duration_minutes,
        COUNT(event_id) AS total_page_views,
        SUM(CASE WHEN event_type IN ('product', 'cart', 'department') THEN 1 ELSE 0 END) AS high_intent_clicks,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS is_purchase_session,
        MAX(CASE WHEN event_type = 'purchase' THEN event_created_at ELSE NULL END) AS purchase_event_time
    FROM Frontend_Traffic_OBT
    GROUP BY session_id
),
intent_tagged AS (
    SELECT 
        session_id,
        user_id,
        session_duration_minutes,
        purchase_event_time,
        CASE 
            WHEN session_duration_minutes < 10 AND total_page_views <= 6 THEN 'Impulse Buyers'
            WHEN session_duration_minutes > 60 OR total_page_views >= 12 THEN 'Heavy Browsers'
            ELSE 'Targeted Searchers'
        END AS user_intent
    FROM session_aggregated
    WHERE is_purchase_session = 1
),
session_with_orders AS (
    SELECT 
        i.user_intent,
        i.session_duration_minutes,
        CASE WHEN o.items_status = 'Returned' THEN 1.0 ELSE 0.0 END AS is_returned
    FROM intent_tagged i
    JOIN Backend_Sales_OBT o ON 
        i.user_id = o.user_id AND 
        o.order_created_at BETWEEN (i.purchase_event_time - INTERVAL 30 MINUTE) AND (i.purchase_event_time + INTERVAL 30 MINUTE)
)
SELECT 
    user_intent AS User_Intent,
    AVG(is_returned) AS Return_Rate,
    AVG(CASE WHEN session_duration_minutes < 120 THEN session_duration_minutes ELSE 120 END) AS Avg_Session_Mins,
    COUNT(*) AS volume
FROM session_with_orders
GROUP BY user_intent
ORDER BY Return_Rate DESC
""")

mart_impulse_penalty.to_parquet('../dataset/processed/mart_impulse_penalty.parquet')

NameError: name 'duckdb' is not defined